# Block 3: Document Parsing with `ai_parse_document`
**Day 1 | 11:00 - 11:55**

---

## What You'll Learn

| Topic | Time |
|---|---|
| Guided: `ai_parse_document` deep-dive | 25 min |
| Hands-on: Parse and explore your own documents | 30 min |

## Why This Matters

In your current architecture, parsing a single document requires **three services working together**:

```
Parsing Service --> Conversion Service --> Step Functions (orchestration)
```

With Databricks, this entire pipeline collapses into **one SQL function call**:

```sql
SELECT ai_parse_document(content, MAP('version', '2.0')) FROM ...
```

No infrastructure to deploy, no Step Functions to maintain, no custom container images to build.

In [0]:
%run ./_resources/Config

---
# Guided Section: `ai_parse_document`
---

## Introduction to `ai_parse_document`

`ai_parse_document` is a built-in Databricks SQL function powered by a **multimodal foundation model**. It extracts structured content from unstructured documents with a single function call.

### What it replaces

| Current Stack | Databricks Equivalent |
|---|---|
| Parsing Service | `ai_parse_document()` |
| Conversion Service | `ai_parse_document()` |
| Step Functions (orchestration) | `ai_parse_document()` |
| Custom infrastructure | **None -- fully managed** |

### Supported file types

| Format | Notes |
|---|---|
| **PDF** | Up to 500 pages per document |
| **PowerPoint** | PPT and PPTX |
| **Word** | DOC and DOCX |
| **Images** | PNG, JPEG, TIFF, BMP, GIF |

### What you get back

The function returns **structured VARIANT output** with element-level detail -- every paragraph, heading, table, and figure is individually identified with its type, content, confidence score, and bounding box coordinates.

> **Presenter Note:** Emphasize the zero-infrastructure angle. They currently maintain container images,
> ECS tasks, and Step Function state machines just to parse documents. All of that goes away.

## Step 1: Parse a PDF Document

Let's start with the simplest possible example -- reading a PDF file and parsing it in a single query.

> **Presenter Note:** Walk through each part of the query:
> - `read_files` with `binaryFile` format reads raw file bytes
> - `pathGlobFilter` selects only PDFs
> - `ai_parse_document` does all the heavy lifting
> - The `MAP('version', '2.0')` parameter selects the latest parser version

In [0]:
display(spark.sql("""
SELECT
  fileName,
  ai_parse_document(
      content, 
      MAP('version', '2.0')) AS parsed
FROM 01_bronze_raw_documents
WHERE fileName = "sample_report_fy2026.pdf"
"""))

path parsed https://databricksinc.sharepoint.com/sites/VantageDemo/Shared%20Documents/short_docs/sample_report_fy2026.pdf {"document":{"elements":[{"bbox":[{"coord":[367,525,1283,611],"page_id":0}],"confidence":0.9999,"content":"Annual Business Report","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[648,681,1006,731],"page_id":0}],"confidence":1,"content":"Fiscal Year 2026","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[611,861,1045,906],"page_id":0}],"confidence":0.9991,"content":"Prepared by: Analytics Team","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[718,948,938,989],"page_id":0}],"confidence":1,"content":"June 17, 2026","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[624,1146,1030,1183],"page_id":0}],"confidence":0.9994,"content":"CONFIDENTIAL - Internal Use Only","description":null,"id":4,"type":"text"},{"bbox":[{"coord":[83,89,515,149],"page_id":1}],"confidence":0.9999,"content":"Executive Summary","description":null,"id":5,"type":"title"},{"bbox":[{"coord":[79,196,1575,384],"page_id":1}],"confidence":0.9995,"content":"This report provides a comprehensive review of business performance for fiscal year 2026. Key highlights include strong\nrevenue growth across all four quarters, with Q4 achieving the highest quarterly revenue in company history.\nOperational efficiencies improved by 12% year-over-year, driven by strategic investments in automation and process\noptimisation.","description":null,"id":6,"type":"text"},{"bbox":[{"coord":[83,439,345,488],"page_id":1}],"confidence":0.9994,"content":"Key Highlights","description":null,"id":7,"type":"section_header"},{"bbox":[{"coord":[79,514,1135,723],"page_id":1}],"confidence":0.9998,"content":"• Total annual revenue exceeded $234.9M, a 23% increase from FY2025.\n• Customer acquisition grew by 18%, reaching 47,200 active accounts.\n• Operating margin improved to 31.4%, up from 28.1% in the prior year.\n• Net Promoter Score (NPS) rose to 72, reflecting improved customer satisfaction.","description":null,"id":8,"type":"text"},{"bbox":[{"coord":[83,770,414,820],"page_id":1}],"confidence":0.9999,"content":"Strategic Priorities","description":null,"id":9,"type":"section_header"},{"bbox":[{"coord":[79,843,1575,984],"page_id":1}],"confidence":0.9997,"content":"Looking ahead to FY2027, the organisation will focus on expanding into two new geographic markets, accelerating\nproduct development cycles, and deepening partnerships with key technology vendors. Investment in data and AI\ncapabilities will remain a top priority to sustain competitive advantage.","description":null,"id":10,"type":"text"},{"bbox":[{"coord":[677,2237,805,2279],"page_id":1}],"confidence":1,"content":"Page 2 of 5","description":null,"id":11,"type":"page_number"},{"bbox":[{"coord":[820,2240,978,2276],"page_id":1}],"confidence":1,"content":"June 17, 2026","description":null,"id":12,"type":"page_footer"},{"bbox":[{"coord":[83,89,498,146],"page_id":2}],"confidence":1,"content":"Financial Summary","description":null,"id":13,"type":"title"},{"bbox":[{"coord":[79,201,1573,287],"page_id":2}],"confidence":0.9999,"content":"The table below summarises quarterly revenue, expenses, and net income for FY2026. All values are reported in USD\nthousands (K).","description":null,"id":14,"type":"text"},{"bbox":[{"coord":[74,334,1578,726],"page_id":2}],"confidence":0.9999,"content":" Quarter Revenue ($K) Expenses ($K) Net Income ($K) Margin (%) Q1 2026 $42,500 $29,200 $13,300 31.3% Q2 2026 $57,800 $38,900 $18,900 32.7% Q3 2026 $63,200 $43,100 $20,100 31.8% Q4 2026 $71,400 $47,800 $23,600 33.1% Full Year $234,900 $159,000 $75,900 32.3% ","description":null,"id":15,"type":"table"},{"bbox":[{"coord":[83,767,626,807],"page_id":2}],"confidence":0.999,"content":"Table 1: Quarterly Financial Summary - FY2026","description":null,"id":16,"type":"caption"},{"bbox":[{"coord":[679,2240,805,2276],"page_id":2}],"confidence":1,"content":"Page 3 of 5","description":null,"id":17,"type":"page_number

### Understanding the Output Structure

The `ai_parse_document` function returns a **VARIANT** (semi-structured JSON) with this schema:

```json
{
  "document": {
    "elements": [
      {
        "type": "text",           // Element classification
        "content": "...",          // The actual extracted content
        "confidence": 0.97,       // Model confidence (0.0 - 1.0)
        "bbox": { ... },          // Bounding box on the page
        "pageId": 0               // Which page this element is on
      }
    ],
    "pages": [
      {
        "pageId": 0,
        "imageUri": "dbfs:/..."   // Only if imageOutputPath is set
      }
    ]
  }
  "error_status": null,          // Any processing errors
  "metadata": {
    "fileName": "...",
    "schemaVersion": "2.0"
  }
}
```

### Element Types

| Type | Description |
|---|---|
| `text` | Body paragraphs and general text |
| `title` | Document title |
| `section_header` | Section and subsection headings |
| `table` | Tables (returned as **HTML**) |
| `figure` | Embedded figures and charts |
| `caption` | Figure/table captions |
| `page_header` | Running headers |
| `page_footer` | Running footers |
| `footnote` | Footnotes |

> **Presenter Note:** Highlight that tables come back as HTML -- this is important because it preserves
> row/column structure. They can parse the HTML downstream or feed it directly to an LLM for extraction.

## Step 2: Explore the Parsed Output Structure

Use VARIANT path notation (the `:` syntax) to pull out specific parts of the parsed output.


In [0]:
display(spark.sql("""
    WITH parsed AS (
    SELECT
        fileName,
        ai_parse_document(
            content, 
            MAP('version', '2.0')) AS doc
    FROM 01_bronze_raw_documents
    WHERE fileName = "sample_report_fy2026.pdf"
    )
    SELECT
        fileName,
        doc:metadata AS metadata,
        size(doc:document:pages::ARRAY<VARIANT>) AS num_pages,
        size(doc:document:elements::ARRAY<VARIANT>) AS num_elements
        FROM parsed
"""))

fileName,metadata,num_pages,num_elements
sample_report_fy2026.pdf,"{""file_metadata"":null,""id"":""0fcf1327-bb9d-422a-ab8f-3f622e493a85"",""version"":""2.0""}",5,37


## Step 3: Extract and Flatten Elements

The real power comes from flattening the `elements` array so each content block becomes its own row.
We use `LATERAL VIEW explode()` to unnest the array.

> **Presenter Note:** This is the key pattern they will use repeatedly:
> 1. Parse the document into VARIANT
> 2. Explode the elements array into rows
> 3. Filter by element type
> 4. Work with the content downstream (chunking, embedding, LLM calls)

In [0]:
display(spark.sql("""
    WITH parsed AS (
    SELECT 
        fileName, 
        ai_parse_document(
            content, 
            MAP('version', '2.0')) AS doc
    FROM 01_bronze_raw_documents
    WHERE fileName = "sample_report_fy2026.pdf"
    )
    SELECT
        fileName,
        elem:type::STRING AS element_type,
        elem:content::STRING AS content,
        elem:confidence::DOUBLE AS confidence
    FROM parsed
        LATERAL VIEW explode(doc:document:elements::ARRAY<VARIANT>) AS elem
    WHERE elem:type::STRING IN ('text', 'title', 'section_header', 'table')
"""))

fileName,element_type,content,confidence
sample_report_fy2026.pdf,title,Annual Business Report,0.9999
sample_report_fy2026.pdf,text,Fiscal Year 2026,0.9999
sample_report_fy2026.pdf,text,Prepared by: Analytics Team,0.9991
sample_report_fy2026.pdf,text,"June 17, 2026",1.0
sample_report_fy2026.pdf,text,CONFIDENTIAL - Internal Use Only,0.9994
sample_report_fy2026.pdf,title,Executive Summary,0.9999
sample_report_fy2026.pdf,text,"This report provides a comprehensive review of business performance for fiscal year 2026. Key highlights include strong revenue growth across all four quarters, with Q4 achieving the highest quarterly revenue in company history. Operational efficiencies improved by 12% year-over-year, driven by strategic investments in automation and process optimisation.",0.9961
sample_report_fy2026.pdf,section_header,Key Highlights,0.9994
sample_report_fy2026.pdf,text,"• Total annual revenue exceeded $234.9M, a 23% increase from FY2025. • Customer acquisition grew by 18%, reaching 47,200 active accounts. • Operating margin improved to 31.4%, up from 28.1% in the prior year. • Net Promoter Score (NPS) rose to 72, reflecting improved customer satisfaction.",0.9997
sample_report_fy2026.pdf,section_header,Strategic Priorities,0.9999


## Step 4: Parse a PowerPoint with Slide Images

One of the most powerful features of `ai_parse_document` is the `imageOutputPath` option.
When set, each slide or page is **rendered as a PNG image** and saved to the specified Unity Catalog Volume.

This directly addresses the requirement:
> *"Each slide outputted as an image"* + *"All text pulled from the slide"*

You get **both** in a single function call.


In [0]:
# Parse a PPTX and output each slide as an image
display(spark.sql(f"""
    WITH parsed AS (
    SELECT 
        fileName, 
        ai_parse_document(
            content,
            MAP(
            'version', '2.0',
            'descriptionElementTypes', 'figure',
            'imageOutputPath', '/Volumes/{catalog}/{schema}/{personal_volume}/slide_images/')
        ) as doc
    FROM 01_bronze_raw_documents
    WHERE fileName = "DatabricksSHORTOverviewDeck.pptx"
    )
    SELECT
        fileName,
        elem:type::STRING AS element_type,
        (elem:bbox[0]:page_id::INT + 1)::INT AS slide_number,
        (elem:bbox[0]:coord::ARRAY<INT>)::STRING AS coordinates,
        elem:description::STRING AS description,
        elem:confidence::DOUBLE AS confidence
    FROM parsed
        LATERAL VIEW explode(doc:document:elements::ARRAY<VARIANT>) AS elem
    WHERE elem:type::STRING IN ('figure')
"""))

fileName,element_type,slide_number,coordinates,description,confidence
DatabricksSHORTOverviewDeck.pptx,figure,1,"[155, 129, 732, 228]","A stylized red and white logo featuring stacked triangles and the word ""databricks"" appears against a dark background.",0.9312
DatabricksSHORTOverviewDeck.pptx,figure,1,"[1422, 127, 2666, 1366]","Four square images of people in office settings are arranged within a larger square, overlaid with geometric shapes.",0.9876
DatabricksSHORTOverviewDeck.pptx,figure,1,"[2443, 1415, 2487, 1455]","A pink, layered, geometric shape resembling stacked steps is centered against a dark background.",0.9993
DatabricksSHORTOverviewDeck.pptx,figure,2,"[1851, 295, 2002, 450]","A large, blue, stylized arrow-like shape points upwards, with a curved line below it.",0.9386
DatabricksSHORTOverviewDeck.pptx,figure,2,"[1250, 536, 1410, 703]","Stacked red trapezoids form a layered, stepped design against a dark background.",0.9999
DatabricksSHORTOverviewDeck.pptx,figure,2,"[2002, 526, 2092, 614]",A central hexagon is surrounded by a symmetrical arrangement of pink squares and yellow triangles.,1.0
DatabricksSHORTOverviewDeck.pptx,figure,2,"[2398, 795, 2538, 956]",A large orange star shape sits above stylized white text on a black background.,0.8876
DatabricksSHORTOverviewDeck.pptx,figure,2,"[1806, 1013, 1952, 1157]","A white, star-shaped design is centered over a square divided into red, yellow, and white sections.",1.0
DatabricksSHORTOverviewDeck.pptx,figure,3,"[420, 325, 637, 539]","A series of horizontal, undulating lines are arranged in a repeating pattern against a solid blue-gray background.",1.0
DatabricksSHORTOverviewDeck.pptx,figure,3,"[955, 325, 1128, 479]",A circular diagram displays a network of interconnected nodes and lines.,0.9999


### Key Point: `imageOutputPath`

When you set `imageOutputPath`:

- Each **slide** (PowerPoint) or **page** (PDF) is rendered as a **PNG image**
- Images are saved to the specified **Unity Catalog Volume** path
- The `pages` array in the output includes `imageUri` fields pointing to each image
- This works for **both** PowerPoint and PDF files

**This directly answers the requirement:**
> "Each slide outputted as an image" + "all text pulled from the slide"

With `ai_parse_document`, you get **text extraction AND image rendering** in one call. No separate
Conversion Service needed.

In **Day 2**, we will send these slide images to multimodal LLMs for visual analysis -- extracting
information from charts, diagrams, and layouts that text-only parsing would miss.

> **Presenter Note:** Connect this forward to Day 2. The generated images are not just for storage --
> they become inputs to vision models. This is the "multimodal pipeline" advantage.

## Step 5: View the Generated Slide Images

Confirm the images were created by listing the output Volume.

In [0]:
# List the generated slide images
display(dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/{personal_volume}/slide_images/"))

path,name,size,modificationTime
dbfs:/Volumes/dennis_schultz/dennis_schultz/my_data/slide_images/2a3f17a1fdd71995be5866efc160a38a46e259412567522df62ee2850210a752.jpg,2a3f17a1fdd71995be5866efc160a38a46e259412567522df62ee2850210a752.jpg,171218,1781817358000
dbfs:/Volumes/dennis_schultz/dennis_schultz/my_data/slide_images/35a5c881a4d99eb35996a0648ac4b2ade853c051c32a89708ffde9fc3acd6aa7.jpg,35a5c881a4d99eb35996a0648ac4b2ade853c051c32a89708ffde9fc3acd6aa7.jpg,234210,1781817357000
dbfs:/Volumes/dennis_schultz/dennis_schultz/my_data/slide_images/ce4ca763b01a8d752f9c56b2a1e544917fedea13fe4a41ce68c242fe609efec0.jpg,ce4ca763b01a8d752f9c56b2a1e544917fedea13fe4a41ce68c242fe609efec0.jpg,89947,1781817358000
dbfs:/Volumes/dennis_schultz/dennis_schultz/my_data/slide_images/faebe6911227757be5a8764dc98f8d57c1feb1b1da49f2acf2744db110afc76a.jpg,faebe6911227757be5a8764dc98f8d57c1feb1b1da49f2acf2744db110afc76a.jpg,163732,1781817357000


## Step 6: Extract Tables from Documents

Tables are returned as **HTML strings**, preserving their row and column structure. This makes them
easy to parse downstream or feed directly to an LLM for structured extraction.

> **Presenter Note:** Show that the HTML output is well-formed. You can render it in a notebook
> with `displayHTML()`, parse it with BeautifulSoup, or pass it directly to an LLM and ask
> "extract the data from this table as JSON."

In [0]:
display(spark.sql("""
    WITH parsed AS (
        SELECT 
            fileName, 
            ai_parse_document(
                content, 
                MAP('version', '2.0')) AS doc
        FROM 01_bronze_raw_documents
        WHERE fileName = "sample_report_fy2026.pdf"
    )
    SELECT
        fileName,
        elem:type::STRING AS element_type,
        elem:content::STRING as table_content -- Tables are returned as HTML
    FROM parsed
        LATERAL VIEW explode(doc:document:elements::ARRAY<VARIANT>) AS elem
    WHERE elem:type::STRING = 'table'
    """)
)

fileName element_type table_content sample_report_fy2026.pdf table Quarter Revenue ($K) Expenses ($K) Net Income ($K) Margin (%) Q1 2026 $42,500 $29,200 $13,300 31.3% Q2 2026 $57,800 $38,900 $18,900 32.7% Q3 2026 $63,200 $43,100 $20,100 31.8% Q4 2026 $71,400 $47,800 $23,600 33.1% Full Year $234,900 $159,000 $75,900 32.3%

## A Note on the Python Approach

You **can** use Python libraries for lower-level parsing if you need very specific control:

| Library | Use Case |
|---|---|
| `python-pptx` | Programmatic PowerPoint manipulation |
| `PyMuPDF` (fitz) | Low-level PDF text/image extraction |
| `python-docx` | Word document manipulation |

**However, `ai_parse_document` is the recommended approach because:**

- Far less code to write and maintain
- Better results -- powered by a multimodal foundation model that understands layout
- No library installation or version management
- Handles edge cases (scanned PDFs, complex layouts) that rule-based parsers miss
- Built-in confidence scores for quality assessment
- Automatic image rendering with `imageOutputPath`

> **Presenter Note:** Acknowledge that some participants may have experience with these libraries.
> The point is not that they are bad tools -- it is that `ai_parse_document` handles 90%+ of use cases
> with a fraction of the code and no infrastructure overhead.

## Step 7: What about Large Documents?
`ai_parse_document` has some [limitations](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document#limitations) that you need to be aware of.  The most significant is that documents are limited to a maximum of 500 pages and 100 MB. The solution is to use the `pageRange` argument to parse the document in chunks.

## Step 7: Parse All Documents and Save to a Delta Table

Parse all documents in the Bronze and persist the results
as a Silver table. This is the pattern you would use in a production pipeline.

**What is happening here:**
1. `spark.read.table(...)` reads the Bronze table into a Dataframe
2. `expr("ai_parse_document(...)")` calls the SQL function from PySpark
3. `.saveAsTable(...)` persists the results as a managed Delta table in Unity Catalog


In [0]:
from pyspark.sql.functions import expr

df = (spark.read.table ("01_bronze_raw_documents"))

df_parsed = df.withColumn(
    "parsed",
    expr(f"""
        ai_parse_document(
            content, 
            MAP('version', '2.0',
            'descriptionElementTypes', 'figure',
            'imageOutputPath', '/Volumes/{catalog}/{schema}/{personal_volume}/slide_images/')
        )
    """)
)

df_parsed.write.mode("overwrite").saveAsTable("02_silver_parsed_documents")
print("Parsed documents saved!")

Parsed documents saved!


## Step 8: Extract All Text from a Specific Document

Take the parsed content of a document from the table and **chunk it for embedding** **_with_** additional context. This is the foundation for building a RAG pipeline -- you need clean, structured text to chunk and embed, but including context such as headers into each chunk will improve accuracy.
 

In [0]:
# ai_prep_search splits the parsed output into semantic chunks ready for RAG --
# no manual element filtering or exploding needed.

display(spark.sql("""
    WITH prepped AS (
        SELECT
            fileName,
            ai_prep_search(parsed) AS result
        FROM 02_silver_parsed_documents
        WHERE path LIKE '%amazon_10-K-2024-As-Filed.pdf'
    )
    SELECT
        fileName,
        chunk:chunk_position::INT       AS chunk_position,
        chunk:chunk_to_retrieve::STRING AS text,
        chunk:chunk_to_embed::STRING    AS text_with_context
    FROM prepped
    LATERAL VIEW explode(result:document:contents::ARRAY<VARIANT>) AS chunk
    ORDER BY chunk_position
"""))

fileName chunk_position text text_with_context amazon_10-K-2024-As-Filed.pdf 0 
Washington, D.C. 20549

(Mark One)
☐ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2024
or
□ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from ___ to __.

Commission File No. 000-22513

(Exact name of registrant as specified in its charter)

Delaware
(State or other jurisdiction of
incorporation or organization)

91-1646860
(I.R.S. Employer
Identification No.)

410 Terry Avenue North
Seattle, Washington 98109-5210
(206) 266-1000

(Address and telephone number, including area code, of registrant’s principal executive offices)

Title of Each Class
Common Stock, par value $.01 per share

Name of Each Exchange on Which Registered
Nasdaq Global Select Market

Trading Symbol(s)
AMZN

None

Indicate by check mark if the registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act. Yes ☒ No □

Indicate by check mark if the registrant is not required to file reports pursuant to Section 13 or Section 15(d) of the Exchange Act. Yes ☐ No ☒

Indicate by check mark whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for
such shorter period that the registrant was required to file such reports), and (2) has been subject to such filing requirements for the past 90 days. Yes ☒ No ☐

Indicate by check mark whether the registrant has submitted electronically every Interactive Data File required to be submitted pursuant to Rule 405 of Regulation S-T during the preceding 12
months (or for such shorter period that the registrant was required to submit such files). Yes ☒ No □

Indicate by check mark whether the registrant is a large accelerated filer, an accelerated filer, a non-accelerated filer, a smaller reporting company, or an emerging growth company. See
definitions of “large accelerated filer,” “accelerated filer,” “smaller reporting company,” and “emerging growth company” in Rule 12b-2 of the Exchange Act.

Large accelerated filer
Non-accelerated filer

Accelerated filer
Smaller reporting company
Emerging growth company

If an emerging growth company, indicate by check mark if the registrant has elected not to use the extended transition period for complying with any new or revised financial accounting
standards provided pursuant to Section 13(a) of the Exchange Act. ☐

Indicate by check mark whether the registrant has filed a report on and attestation to its management’s assessment of the effectiveness of its internal control over financial reporting under Section
404(b) of the Sarbanes-Oxley Act (15 U.S.C. 7262(b)) by the registered public accounting firm that prepared or issued its audit report. ☒

If securities are registered pursuant to Section 12(b) of the Exchange Act, indicate by check mark whether the financial statements of the registrant included in the filing reflect the correction of
an error to previously issued financial statements. □

Indicate by check mark whether any of those error corrections are restatements that required a recovery analysis of incentive-based compensation received by any of the registrant’s executive
officers during the relevant recovery period pursuant to §240.10D-1(b). ☐

Indicate by check mark whether the registrant is a shell company (as defined in Rule 12b-2 of the Exchange Act). Yes □ No ☒

Aggregate market value of voting stock held by non-affiliates of the registrant as of June 30, 2024

$ 1,815,014,489,485
10,597,729,352

Number of shares of common stock outstanding as of January 29, 2025

The information required by Part III of this Report, to the extent not set forth herein, is incorporated herein by reference from the registrant's definitive proxy statement relating to the Annual
Meeting of Shareholders to be held in 2025, which

---
## Recap: What We Covered

### Key Takeaways

1. **`ai_parse_document`** is a single SQL function that replaces your Parsing Service, Conversion Service, and Step Functions orchestration
2. It returns **structured VARIANT output** with element-level classification (text, table, figure, etc.)
3. Tables are extracted as **HTML**, preserving structure for downstream processing
4. The **`imageOutputPath`** parameter renders slides/pages as PNG images
4. The **descriptionElementsTypes** provides an AI generated description for any figures and images in the document
5. The same function works from **SQL and PySpark** via `expr()`
6. Results persist naturally in **Delta tables** for reuse across the pipeline

### Architecture Impact

```
BEFORE (Current State):
  S3 Upload -> Lambda -> Step Functions -> Parsing Service -> Conversion Service -> S3
  (6 components, multiple failure points, custom error handling)

AFTER (Databricks):
  SharePoint -> Bronze -> ai_parse_document() -> Silver Table
  (1 function call, automatic retries, governed by Unity Catalog)
```

### What's Next

In Day 2, we will use the slide images with multimodal LLMs.